# Reference-guided Video Generation on CUDA T4 (Colab)

Run [`video-gen`](../src/video_gen/generate.py) with **Wan2.1-VACE-14B** (LightX2V step distillation merged in, GGUF `Q3_K_M`) on a T4.
Give it subject/style **reference images** and a **prompt**; it produces a **4-second, 1280x720 (16:9)** video.
Google Drive is used **only** for the reference images and the output videos.

**Runtime:** Runtime → Change runtime type → GPU (T4).

All models are public, so no Hugging Face token is required (setting `HF_TOKEN` only speeds up downloads).

## 1. Clone the repo

In [ ]:
# Set your GitHub URL after pushing this repo, then run.
REPO_URL = "https://github.com/whizsid/video_generation.git"  # <-- edit me
REPO_DIR = "/content/video_generation"

import os
from pathlib import Path

if Path(REPO_DIR).exists():
    print(f"Already cloned at {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("cwd:", os.getcwd())

## 2. Install pip and system dependencies

In [ ]:
# ffmpeg for video encoding; project deps (diffusers main, transformers, gguf, spandrel).
# Colab ships CUDA torch — keep it.
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null
!pip install -q -e .
# Colab's preinstalled torchao is older than diffusers main expects and breaks `import diffusers`;
# this project quantizes via GGUF and doesn't need it.
!pip uninstall -y -q torchao

print("Install done.")

## 3. Mount Google Drive

Put your reference images (`.png`, `.jpg`, `.jpeg`, `.webp`) in `MyDrive/<WORK_REL_DIR>/refs/`.
Videos are written to `MyDrive/<WORK_REL_DIR>/outputs/`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# @markdown Folder relative to `MyDrive` (e.g. `video_gen`).
WORK_REL_DIR = "video_gen"  # @param {type:"string"}

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_DIR = DRIVE_ROOT / WORK_REL_DIR.strip().strip("/")
REFS_DIR = DRIVE_DIR / "refs"
OUTPUT_DIR = DRIVE_DIR / "outputs"

REFS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
drive_refs = sorted(p for p in REFS_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS)
print("REFS_DIR:", REFS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print(f"Reference images ({len(drive_refs)}):", [p.name for p in drive_refs])

## 4. Check runtime

In [ ]:
import shutil
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → T4)."
idx = torch.cuda.current_device()
name = torch.cuda.get_device_name(idx)
props = torch.cuda.get_device_properties(idx)
free, total = torch.cuda.mem_get_info(idx)
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"GPU: {name}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB total, "
      f"{free / 1024**3:.1f}/{total / 1024**3:.1f} GiB free")
print(f"System RAM: {ram_gib:.1f} GiB | free disk: {disk_free_gib:.0f} GiB")
print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if "T4" not in name:
    print(f"Note: expected a T4; got {name!r}. With more VRAM/RAM you can raise QUANT and RESOLUTION below.")
if disk_free_gib < 30:
    print("Warning: less than 30 GiB free disk; the weights need ~21 GiB.")

## 5. Pull Wan2.1-VACE-14B weights

~21 GB into the local Hugging Face cache: the GGUF transformer, the UMT5-XXL text encoder, the VAE and configs.
Pick the quantization here; the run cell uses the same value.

In [ ]:
import os
import sys
from huggingface_hub import hf_hub_download, snapshot_download

# The editable install is only picked up by new processes; make it importable in this kernel too.
sys.path.insert(0, f"{REPO_DIR}/src")
from video_gen.config import GGUF_FILE_TEMPLATE, GGUF_REPO_14B, MODEL_ID_14B

# @markdown Q3_K_M is the largest quant that fits free-Colab RAM and T4 VRAM.
QUANT = "Q3_K_M"  # @param ["Q3_K_S", "Q3_K_M", "Q3_K_L", "Q4_K_S", "Q4_K_M", "Q5_K_M", "Q8_0"]

HF_TOKEN = os.environ.get("HF_TOKEN", "")  # optional: HF_TOKEN = "hf_..."
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

base = snapshot_download(
    MODEL_ID_14B,
    allow_patterns=["model_index.json", "scheduler/*", "text_encoder/*", "tokenizer/*", "vae/*",
                    "transformer/config.json"],
)
gguf = hf_hub_download(GGUF_REPO_14B, GGUF_FILE_TEMPLATE.format(quant=QUANT))

print("Base components:", base)
print("Transformer:", gguf, f"({os.path.getsize(gguf) / 1e9:.2f} GB)")

## 6. Small smoke test (optional)

A ~1-second, 288p clip from the prompt alone, without upscaling, to check the setup end to end.
Set `RUN_SMOKE = True` to run it; otherwise go straight to the next cell.

In [ ]:
from pathlib import Path

RUN_SMOKE = False  # @param {type:"boolean"}

if not RUN_SMOKE:
    print("Skipping smoke test — run the Drive cell below.")
else:
    !video-gen --profile t4 \
      --prompt "A red apple on a wooden table, soft window light, photorealistic" \
      --quant {QUANT} \
      --resolution 288p \
      --frames 17 \
      --steps 4 \
      --no-interpolate \
      --upscale none \
      --seed 0 \
      --out outputs/smoke.mp4
    smoke = Path("outputs/smoke.mp4")
    print("Smoke output:", smoke, "exists=", smoke.exists())
    assert smoke.exists(), "Smoke test produced no video"

## 7. Run on Drive references → save output to Drive

Describe subject, action, setting, lighting and camera. Gentle, clear motion gives the most realistic results.

- `STEPS`: 6 by default; 4 is faster, 8 rarely helps with this distilled model.
- `RESOLUTION`: `432p` (768x432) is the largest that fits on a T4; the result is upscaled afterwards.
- `REF_STRENGTH`: raise (e.g. 1.2) to follow the references more closely.

In [ ]:
import shlex
import shutil
from datetime import datetime
from pathlib import Path

from IPython.display import Video, display

PROMPT = "A young woman with curly hair walks slowly along a sunlit beach at golden hour, gentle waves, handheld camera, shallow depth of field, photorealistic"  # @param {type:"string"}
SEED = 42  # @param {type:"integer"}
SECONDS = 4  # @param {type:"number"}
STEPS = 6  # @param {type:"integer"}
RESOLUTION = "432p"  # @param ["288p", "432p"]
UPSCALE = "720p"  # @param ["none", "720p", "1080p"]
REF_STRENGTH = 1.0  # @param {type:"number"}
OUTPUT_NAME = ""  # @param {type:"string"}

# Copy references off Drive onto local Colab disk so loading never waits on Drive I/O.
drive_refs = sorted(p for p in REFS_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS)
LOCAL_REFS_DIR = Path("/content/refs")
LOCAL_REFS_DIR.mkdir(exist_ok=True)
local_refs = []
for src in drive_refs:
    dst = LOCAL_REFS_DIR / src.name
    if (not dst.exists()) or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
    local_refs.append(dst)
if local_refs:
    print("Local references:", [p.name for p in local_refs])
else:
    print(f"Warning: no images in {REFS_DIR}; generating from the prompt alone.")

# Write locally first, then copy both videos to Drive.
stem = OUTPUT_NAME.strip() or f"clip_{datetime.now():%Y%m%d_%H%M%S}"
local_out = Path("outputs") / f"{stem}.mp4"

args = ["video-gen", "--profile", "t4", "--prompt", PROMPT, "--seed", SEED, "--seconds", SECONDS,
        "--quant", QUANT, "--steps", STEPS, "--resolution", RESOLUTION, "--upscale", UPSCALE,
        "--ref-strength", REF_STRENGTH, "--out", local_out]
for ref in local_refs:
    args += ["--ref", ref]
cmd = " ".join(shlex.quote(str(a)) for a in args)
print(cmd)
!{cmd}

assert local_out.exists(), f"Missing output {local_out}"
saved = []
for path in (local_out, local_out.with_name(f"{stem}_raw.mp4")):
    if path.exists():
        shutil.copy2(path, OUTPUT_DIR / path.name)
        saved.append(OUTPUT_DIR / path.name)
print("Saved to Drive:", *saved, sep="\n  ")

display(Video(str(local_out), embed=True, width=960))